In [25]:

from langgraph.graph import StateGraph , START, END
from langchain_groq import ChatGroq
from typing import List, Dict, Any, TypedDict
from dotenv import load_dotenv
import os


In [26]:
load_dotenv()  # Load environment variables from .env file

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ.get("GROQ_API_KEY")
)

In [32]:
 #defined the state
class PromptChaining(TypedDict):
    topic: str
    outline: str
    blog: str
    evaluate: str

In [33]:
# node 1
def outline_generator(state: PromptChaining) -> PromptChaining:
    topic = state['topic']
    prompt = f"Write an outline for a blog post about {topic}"
    
    response = model.invoke(prompt)
    
    outline = response.content
    
    state['outline'] = outline
    
    return state


def blog_generator(state: PromptChaining) -> PromptChaining:
    outline = state['outline']
    
    prompt = f"Write a blog post based on the following outline: {outline}"
    
    response = model.invoke(prompt)
    
    blog = response.content
    
    state['blog'] = blog
    
    return state


def blog_evaluator(state: PromptChaining) -> PromptChaining:
    blog = state['blog']
    
    prompt = f"Evaluate the following blog post and provide feedback and give me rating for this blog: {blog}"
    
    response = model.invoke(prompt)
    
    evaluation = response.content
    
    state['evaluate'] = evaluation
    
    return state
    
    
    
  

    

In [34]:
# create a graph 
graph = StateGraph(PromptChaining)


# craate a nodes
graph.add_node('outline', outline_generator)
graph.add_node('blog', blog_generator)
graph.add_node('evaluate', blog_evaluator)

# create a edges

graph.add_edge(START, 'outline')
graph.add_edge('outline', 'blog')
graph.add_edge('blog', 'evaluate')
graph.add_edge('evaluate', END)


# compile the graph

graph.compile()
workflow = graph.compile()



In [35]:
initial_state = {'topic': 'write a blog on how coding works'}

result = workflow.invoke(initial_state)

print(result)

{'topic': 'write a blog on how coding works', 'outline': '**Title:** *How Coding Actually Works – A Beginner‑Friendly Walk‑through*\n\n---\n\n### 1. Introduction  \n- **Hook:** A relatable anecdote (e.g., “Ever wondered how typing a few lines turns into a game, an app, or a website?”)  \n- **Why it matters:** Understanding the mechanics behind code demystifies technology and empowers anyone to create.  \n- **What the reader will learn:** The journey from human‑readable instructions to machine execution, key concepts, and practical take‑aways.\n\n---\n\n### 2. What Is “Coding” Anyway?  \n- **Definition:** Translating human logic into a language a computer can understand.  \n- **Analogy:** Compare coding to giving a recipe to a robot chef.  \n- **Common misconceptions** (e.g., “coding = just typing”; “only for math geniuses”).\n\n---\n\n### 3. The Building Blocks of Code  \n#### 3.1. Syntax & Semantics  \n- *Syntax:* Grammar rules (e.g., brackets, semicolons).  \n- *Semantics:* Meaning b

In [37]:
print(result['evaluate'])

**Overall Rating: 8 / 10**  

| Aspect | Score | Comments |
|--------|------|----------|
| **Clarity & Readability** | 8 | The post reads like a conversation with the reader, uses analogies (“recipe for a robot chef”) and plenty of visual breaks (tables, code blocks). That makes it very approachable for beginners. |
| **Technical Accuracy** | 9 | All core concepts (syntax vs. semantics, compilation pipeline, JIT, control‑flow constructs, data types, paradigms) are explained correctly. The mini‑project works as‑is and demonstrates the full “source → execution” cycle. |
| **Depth vs. Audience** | 7 | It covers a surprisingly wide range of topics for a “beginner‑friendly” piece (e.g., AST, linking, JIT, closures). Most beginners will appreciate the breadth, but a few sections (the compilation pipeline, JIT) might feel a bit heavy without additional visual aids. |
| **Structure & Flow** | 8 | Logical progression: definition → building blocks → how code runs → paradigms → tools → hands‑on e